# SQL Data Analysis | Project 3

**DecodeLabs Industrial Training**

This notebook demonstrates SQL querying skills using Python's sqlite3 module on an e-commerce orders dataset. The dataset contains 1200 orders with information about products and customers and payments and order statuses.

**Objectives:**
- Write SELECT queries to retrieve specific data
- Use WHERE for filtering and ORDER BY for sorting and GROUP BY for aggregation
- Perform aggregations using COUNT and SUM and AVG
- Use HAVING to filter grouped results
- Calculate percentage contributions of categories

## 1. Setup and Data Loading

Load the cleaned dataset into an in-memory SQLite database so we can run SQL queries against it.

In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

# Load the CSV into a pandas DataFrame
df = pd.read_csv('Cleaned_Dataset.csv')

# Create an in-memory SQLite database and load the data
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

print(f'Database created successfully with {len(df)} records')
print(f'Table: orders')
print(f'Columns: {list(df.columns)}')

Database created successfully with 1200 records
Table: orders
Columns: ['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice']


We define a helper function to execute SQL queries and return the results as a clean DataFrame.

In [2]:
def run_query(query):
    """Execute a SQL query and return results as a pandas DataFrame."""
    return pd.read_sql_query(query, conn)

## 2. Exploring the Table Schema

Before running analytical queries we inspect the table structure and a sample of the data.

In [3]:
# View the table schema
schema = run_query("PRAGMA table_info(orders)")
print('Table Schema:')
schema

Table Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,OrderID,TEXT,0,None,0
1,1,Date,TEXT,0,None,0
2,2,CustomerID,TEXT,0,None,0
3,3,Product,TEXT,0,None,0
4,4,Quantity,INTEGER,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,ShippingAddress,TEXT,0,None,0
7,7,PaymentMethod,TEXT,0,None,0
8,8,OrderStatus,TEXT,0,None,0
9,9,TrackingNumber,TEXT,0,None,0


In [4]:
# Preview the first 10 rows
result = run_query("SELECT * FROM orders LIMIT 10")
result

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
5,ORD200005,2023-10-23,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72
6,ORD200006,2025-06-17,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42
7,ORD200007,2023-05-12,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,NoCoupon,Email,268.56
9,ORD200009,2023-11-21,C31946,Desk,4,509.38,102 Main St,Credit Card,Shipped,TRK33478363,6,SAVE10,Google,2037.52


In [5]:
# Total number of records in the table
result = run_query("SELECT COUNT(*) AS TotalOrders FROM orders")
result

,TotalOrders
0,1200


## 3. SELECT Queries with Filtering (WHERE)

The WHERE clause lets us filter rows based on conditions. This is fundamental to extracting specific subsets of data.

### 3.1 Orders with a Total Price Above 2000

In [6]:
# Select high value orders (TotalPrice > 2000)
result = run_query("""
    SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice
    FROM orders
    WHERE TotalPrice > 2000
    ORDER BY TotalPrice DESC
    LIMIT 15
""")
print(f'Total high value orders (above 2000): {len(run_query("SELECT * FROM orders WHERE TotalPrice > 2000"))}')
result

Total high value orders (above 2000): 180


,OrderID,Product,Quantity,UnitPrice,TotalPrice
0,ORD200789,Tablet,5,691.28,3456.40
1,ORD201122,Monitor,5,678.19,3390.95
2,ORD200632,Laptop,5,678.16,3390.80
3,ORD200469,Chair,5,676.98,3384.90
4,ORD200328,Tablet,5,674.04,3370.20
5,ORD200107,Printer,5,670.75,3353.75
6,ORD200326,Laptop,5,670.48,3352.40
7,ORD201065,Printer,5,666.80,3334.00
8,ORD201031,Phone,5,664.51,3322.55
9,ORD200463,Laptop,5,662.78,3313.90


### 3.2 Orders for a Specific Product (Laptop)

In [7]:
# Select all Laptop orders with their details
result = run_query("""
    SELECT OrderID, Date, CustomerID, Quantity, UnitPrice, TotalPrice, OrderStatus
    FROM orders
    WHERE Product = 'Laptop'
    ORDER BY TotalPrice DESC
    LIMIT 15
""")
print(f'Total Laptop orders: {len(run_query("SELECT * FROM orders WHERE Product = \'Laptop\'"))}')
result

Total Laptop orders: 173


,OrderID,Date,CustomerID,Quantity,UnitPrice,TotalPrice,OrderStatus
0,ORD200632,2023-05-02,C67260,5,678.16,3390.80,Delivered
1,ORD200326,2024-07-01,C65986,5,670.48,3352.40,Returned
2,ORD200463,2023-05-26,C25276,5,662.78,3313.90,Shipped
3,ORD200367,2024-04-25,C13108,5,658.77,3293.85,Pending
4,ORD200540,2024-01-29,C87281,5,648.65,3243.25,Pending
5,ORD200764,2024-03-10,C35983,5,627.43,3137.15,Cancelled
6,ORD200492,2023-10-25,C39074,5,606.52,3032.60,Shipped
7,ORD200633,2024-04-10,C79533,5,601.72,3008.60,Cancelled
8,ORD201087,2025-03-23,C84134,4,693.07,2772.28,Shipped
9,ORD201156,2023-07-19,C20512,4,690.78,2763.12,Shipped


### 3.3 Cancelled Orders with a High Quantity (4 or More)

In [8]:
# Cancelled orders where quantity is 4 or more
result = run_query("""
    SELECT OrderID, Product, Quantity, TotalPrice, PaymentMethod
    FROM orders
    WHERE OrderStatus = 'Cancelled' AND Quantity >= 4
    ORDER BY TotalPrice DESC
    LIMIT 15
""")
print(f'Total cancelled orders with quantity >= 4: {len(run_query("SELECT * FROM orders WHERE OrderStatus = \'Cancelled\' AND Quantity >= 4"))}')
result

Total cancelled orders with quantity >= 4: 105


,OrderID,Product,Quantity,TotalPrice,PaymentMethod
0,ORD200469,Chair,5,3384.90,Cash
1,ORD200328,Tablet,5,3370.20,Online
2,ORD200527,Chair,5,3267.35,Credit Card
3,ORD200768,Tablet,5,3267.30,Cash
4,ORD200889,Monitor,5,3253.60,Credit Card
5,ORD200802,Chair,5,3223.20,Gift Card
6,ORD200086,Printer,5,3215.15,Online
7,ORD200296,Desk,5,3194.00,Debit Card
8,ORD200364,Phone,5,3143.70,Cash
9,ORD200764,Laptop,5,3137.15,Credit Card


### 3.4 Orders Placed in the Year 2025

In [9]:
# Orders from the year 2025
result = run_query("""
    SELECT OrderID, Date, Product, TotalPrice, OrderStatus
    FROM orders
    WHERE Date LIKE '2025%'
    ORDER BY Date
    LIMIT 15
""")
print(f'Total orders in 2025: {len(run_query("SELECT * FROM orders WHERE Date LIKE \'2025%\'"))}')
result

Total orders in 2025: 231


,OrderID,Date,Product,TotalPrice,OrderStatus
0,ORD200408,2025-01-01,Printer,1237.59,Pending
1,ORD200152,2025-01-02,Chair,798.08,Cancelled
2,ORD201147,2025-01-02,Chair,1176.60,Pending
3,ORD200211,2025-01-03,Printer,580.74,Pending
4,ORD200438,2025-01-03,Tablet,1378.02,Pending
5,ORD200168,2025-01-04,Printer,594.96,Returned
6,ORD200595,2025-01-10,Printer,770.32,Pending
7,ORD201043,2025-01-11,Tablet,359.92,Shipped
8,ORD200292,2025-01-15,Monitor,65.95,Delivered
9,ORD200375,2025-01-16,Monitor,380.12,Shipped


### 3.5 Orders Using Gift Card Payment with FREESHIP Coupon

In [10]:
# Gift card orders with FREESHIP coupon
result = run_query("""
    SELECT OrderID, Product, TotalPrice, OrderStatus, ReferralSource
    FROM orders
    WHERE PaymentMethod = 'Gift Card' AND CouponCode = 'FREESHIP'
    ORDER BY TotalPrice DESC
    LIMIT 15
""")
print(f'Total Gift Card orders with FREESHIP: {len(run_query("SELECT * FROM orders WHERE PaymentMethod = \'Gift Card\' AND CouponCode = \'FREESHIP\'"))}')
result

Total Gift Card orders with FREESHIP: 72


,OrderID,Product,TotalPrice,OrderStatus,ReferralSource
0,ORD200107,Printer,3353.75,Shipped,Instagram
1,ORD200367,Laptop,3293.85,Pending,Instagram
2,ORD200802,Chair,3223.20,Cancelled,Email
3,ORD200252,Phone,2673.44,Cancelled,Facebook
4,ORD200781,Phone,2621.30,Delivered,Referral
5,ORD200587,Monitor,2573.00,Delivered,Instagram
6,ORD200282,Printer,2551.16,Cancelled,Google
7,ORD200118,Printer,2513.55,Shipped,Google
8,ORD200345,Tablet,2506.30,Shipped,Referral
9,ORD200965,Laptop,2313.12,Cancelled,Email


## 4. Sorting with ORDER BY

ORDER BY allows us to sort query results in ascending or descending order based on one or more columns.

### 4.1 Top 10 Most Expensive Orders

In [11]:
# Top 10 orders by total price
result = run_query("""
    SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice, OrderStatus
    FROM orders
    ORDER BY TotalPrice DESC
    LIMIT 10
""")
result

,OrderID,Product,Quantity,UnitPrice,TotalPrice,OrderStatus
0,ORD200789,Tablet,5,691.28,3456.40,Delivered
1,ORD201122,Monitor,5,678.19,3390.95,Returned
2,ORD200632,Laptop,5,678.16,3390.80,Delivered
3,ORD200469,Chair,5,676.98,3384.90,Cancelled
4,ORD200328,Tablet,5,674.04,3370.20,Cancelled
5,ORD200107,Printer,5,670.75,3353.75,Shipped
6,ORD200326,Laptop,5,670.48,3352.40,Returned
7,ORD201065,Printer,5,666.80,3334.00,Delivered
8,ORD201031,Phone,5,664.51,3322.55,Pending
9,ORD200463,Laptop,5,662.78,3313.90,Shipped


### 4.2 Top 10 Cheapest Orders

In [12]:
# Top 10 cheapest orders
result = run_query("""
    SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice, OrderStatus
    FROM orders
    ORDER BY TotalPrice ASC
    LIMIT 10
""")
result

,OrderID,Product,Quantity,UnitPrice,TotalPrice,OrderStatus
0,ORD201161,Phone,1,11.39,11.39,Cancelled
1,ORD200863,Phone,1,14.06,14.06,Pending
2,ORD200240,Tablet,1,17.24,17.24,Pending
3,ORD200542,Tablet,1,17.98,17.98,Cancelled
4,ORD200336,Laptop,1,18.20,18.20,Pending
5,ORD200776,Tablet,1,21.19,21.19,Returned
6,ORD201025,Chair,1,23.53,23.53,Delivered
7,ORD200690,Monitor,1,24.48,24.48,Returned
8,ORD200926,Desk,1,26.95,26.95,Delivered
9,ORD200473,Chair,2,14.93,29.86,Shipped


### 4.3 Most Recent Orders

In [13]:
# 10 most recent orders sorted by date
result = run_query("""
    SELECT OrderID, Date, Product, CustomerID, TotalPrice, OrderStatus
    FROM orders
    ORDER BY Date DESC
    LIMIT 10
""")
result

,OrderID,Date,Product,CustomerID,TotalPrice,OrderStatus
0,ORD200256,2025-06-30,Chair,C94323,379.05,Returned
1,ORD201107,2025-06-30,Tablet,C25110,126.42,Cancelled
2,ORD200176,2025-06-28,Tablet,C80151,1723.32,Pending
3,ORD200773,2025-06-28,Phone,C99205,1024.66,Cancelled
4,ORD200882,2025-06-28,Phone,C43335,548.24,Delivered
5,ORD201039,2025-06-28,Desk,C49137,598.46,Cancelled
6,ORD200226,2025-06-27,Desk,C31501,1764.15,Delivered
7,ORD200064,2025-06-25,Printer,C49720,391.83,Cancelled
8,ORD200055,2025-06-24,Printer,C95984,1033.20,Cancelled
9,ORD200163,2025-06-24,Tablet,C99589,641.39,Cancelled


### 4.4 Orders Sorted by Product Name and Then by Quantity

In [14]:
# Multi column sort: by product alphabetically then by quantity descending
result = run_query("""
    SELECT OrderID, Product, Quantity, TotalPrice
    FROM orders
    ORDER BY Product ASC, Quantity DESC
    LIMIT 15
""")
result

,OrderID,Product,Quantity,TotalPrice
0,ORD200020,Chair,5,312.55
1,ORD200066,Chair,5,635.90
2,ORD200175,Chair,5,341.70
3,ORD200177,Chair,5,2158.45
4,ORD200241,Chair,5,3078.35
5,ORD200256,Chair,5,379.05
6,ORD200267,Chair,5,2342.55
7,ORD200277,Chair,5,454.60
8,ORD200352,Chair,5,869.80
9,ORD200378,Chair,5,322.50


## 5. Aggregations with GROUP BY

GROUP BY combined with aggregate functions like COUNT and SUM and AVG allows us to summarize data by categories.

### 5.1 Order Count by Product

In [15]:
# Number of orders per product
result = run_query("""
    SELECT Product,
           COUNT(*) AS OrderCount
    FROM orders
    GROUP BY Product
    ORDER BY OrderCount DESC
""")
result

,Product,OrderCount
0,Printer,181
1,Tablet,179
2,Chair,178
3,Laptop,173
4,Desk,170
5,Monitor,163
6,Phone,156


### 5.2 Total Revenue by Product

In [16]:
# Total revenue generated by each product
result = run_query("""
    SELECT Product,
           COUNT(*) AS OrderCount,
           SUM(TotalPrice) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
    FROM orders
    GROUP BY Product
    ORDER BY TotalRevenue DESC
""")
result

,Product,OrderCount,TotalRevenue,AvgOrderValue
0,Chair,178,195620.11,1098.99
1,Printer,181,195612.61,1080.73
2,Laptop,173,192126.56,1110.56
3,Tablet,179,186568.95,1042.28
4,Monitor,163,175651.41,1077.62
5,Desk,170,167459.93,985.06
6,Phone,156,151722.39,972.58


### 5.3 Order Count and Revenue by Order Status

In [17]:
# Breakdown by order status
result = run_query("""
    SELECT OrderStatus,
           COUNT(*) AS OrderCount,
           SUM(TotalPrice) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           SUM(Quantity) AS TotalUnitsSold
    FROM orders
    GROUP BY OrderStatus
    ORDER BY OrderCount DESC
""")
result

,OrderStatus,OrderCount,TotalRevenue,AvgOrderValue,TotalUnitsSold
0,Cancelled,250,276396.21,1105.58,759
1,Returned,247,243277.70,984.93,682
2,Pending,237,256328.15,1081.55,724
3,Shipped,235,246159.58,1047.49,680
4,Delivered,231,242600.32,1050.22,690


### 5.4 Revenue by Payment Method

In [18]:
# Revenue and order count by payment method
result = run_query("""
    SELECT PaymentMethod,
           COUNT(*) AS OrderCount,
           SUM(TotalPrice) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           MIN(TotalPrice) AS MinOrder,
           MAX(TotalPrice) AS MaxOrder
    FROM orders
    GROUP BY PaymentMethod
    ORDER BY TotalRevenue DESC
""")
result

,PaymentMethod,OrderCount,TotalRevenue,AvgOrderValue,MinOrder,MaxOrder
0,Credit Card,234,263847.63,1127.55,14.06,3299.25
1,Online,258,262442.94,1017.22,17.24,3456.40
2,Cash,246,259786.29,1056.04,11.39,3384.90
3,Gift Card,230,246323.92,1070.97,23.53,3390.80
4,Debit Card,232,232361.18,1001.56,18.20,3334.00


### 5.5 Performance by Referral Source

In [19]:
# Revenue and average order value by referral source
result = run_query("""
    SELECT ReferralSource,
           COUNT(*) AS OrderCount,
           SUM(TotalPrice) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           ROUND(AVG(Quantity), 2) AS AvgQuantity
    FROM orders
    GROUP BY ReferralSource
    ORDER BY TotalRevenue DESC
""")
result

,ReferralSource,OrderCount,TotalRevenue,AvgOrderValue,AvgQuantity
0,Instagram,259,275285.45,1062.88,2.97
1,Email,250,261808.55,1047.23,2.98
2,Google,241,250441.48,1039.18,2.96
3,Facebook,228,250410.90,1098.29,2.96
4,Referral,222,226815.58,1021.69,2.84


### 5.6 Coupon Code Usage Analysis

In [20]:
# Order count and revenue by coupon code
result = run_query("""
    SELECT CouponCode,
           COUNT(*) AS OrderCount,
           SUM(TotalPrice) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           SUM(Quantity) AS TotalUnitsSold
    FROM orders
    GROUP BY CouponCode
    ORDER BY TotalRevenue DESC
""")
result

,CouponCode,OrderCount,TotalRevenue,AvgOrderValue,TotalUnitsSold
0,FREESHIP,313,335036.99,1070.41,914
1,NoCoupon,309,322401.41,1043.37,941
2,SAVE10,286,304840.02,1065.87,844
3,WINTER15,292,302483.54,1035.90,836


### 5.7 Monthly Revenue Trend

In [21]:
# Revenue trend by year and month
result = run_query("""
    SELECT SUBSTR(Date, 1, 7) AS YearMonth,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
    FROM orders
    GROUP BY YearMonth
    ORDER BY YearMonth
""")
result

,YearMonth,OrderCount,TotalRevenue,AvgOrderValue
0,2023-01,47,56685.75,1206.08
1,2023-02,37,40117.66,1084.26
2,2023-03,43,48609.37,1130.45
3,2023-04,31,27751.71,895.22
4,2023-05,49,63836.84,1302.79
5,2023-06,45,49500.19,1100.00
6,2023-07,44,42820.66,973.20
7,2023-08,51,54352.14,1065.73
8,2023-09,29,29526.67,1018.16
9,2023-10,47,52607.85,1119.32


### 5.8 Yearly Revenue Summary

In [22]:
# Revenue summary by year
result = run_query("""
    SELECT SUBSTR(Date, 1, 4) AS Year,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           SUM(Quantity) AS TotalUnitsSold
    FROM orders
    GROUP BY Year
    ORDER BY Year
""")
result

,Year,OrderCount,TotalRevenue,AvgOrderValue,TotalUnitsSold
0,2023,510,552643.24,1083.61,1524
1,2024,459,480235.87,1046.27,1351
2,2025,231,231882.85,1003.82,660


## 6. HAVING: Filtering Grouped Data

While WHERE filters individual rows before grouping the HAVING clause filters groups after aggregation. This is particularly useful for finding categories that meet certain thresholds.

### 6.1 Products with Total Revenue Exceeding 150000

In [23]:
# Products where total revenue exceeds 150000
result = run_query("""
    SELECT Product,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
    FROM orders
    GROUP BY Product
    HAVING TotalRevenue > 150000
    ORDER BY TotalRevenue DESC
""")
print('Products with revenue above 150000:')
result

Products with revenue above 150000:


,Product,OrderCount,TotalRevenue,AvgOrderValue
0,Chair,178,195620.11,1098.99
1,Printer,181,195612.61,1080.73
2,Laptop,173,192126.56,1110.56
3,Tablet,179,186568.95,1042.28
4,Monitor,163,175651.41,1077.62
5,Desk,170,167459.93,985.06
6,Phone,156,151722.39,972.58


### 6.2 Payment Methods with More Than 200 Orders

In [24]:
# Payment methods used in more than 200 orders
result = run_query("""
    SELECT PaymentMethod,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
    FROM orders
    GROUP BY PaymentMethod
    HAVING OrderCount > 200
    ORDER BY OrderCount DESC
""")
print('Payment methods with more than 200 orders:')
result

Payment methods with more than 200 orders:


,PaymentMethod,OrderCount,TotalRevenue,AvgOrderValue
0,Online,258,262442.94,1017.22
1,Cash,246,259786.29,1056.04
2,Credit Card,234,263847.63,1127.55
3,Debit Card,232,232361.18,1001.56
4,Gift Card,230,246323.92,1070.97


### 6.3 Customers Who Placed More Than One Order

In [25]:
# Repeat customers (more than 1 order)
result = run_query("""
    SELECT CustomerID,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalSpent,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
    FROM orders
    GROUP BY CustomerID
    HAVING OrderCount > 1
    ORDER BY TotalSpent DESC
""")
print(f'Repeat customers: {len(result)}')
result.head(10)

Repeat customers: 11


,CustomerID,OrderCount,TotalSpent,AvgOrderValue
0,C38840,2,5723.23,2861.61
1,C97593,2,2855.22,1427.61
2,C98474,2,2175.34,1087.67
3,C70659,2,1853.96,926.98
4,C94569,2,1675.35,837.68
5,C46651,2,1360.89,680.44
6,C35852,2,1248.39,624.20
7,C14847,2,1097.81,548.90
8,C91155,2,704.82,352.41
9,C21191,2,647.77,323.88


### 6.4 Products with an Average Order Value Above 1200

In [26]:
# Products with average order value exceeding 1200
result = run_query("""
    SELECT Product,
           COUNT(*) AS OrderCount,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           ROUND(AVG(Quantity), 2) AS AvgQuantity,
           ROUND(AVG(UnitPrice), 2) AS AvgUnitPrice
    FROM orders
    GROUP BY Product
    HAVING AvgOrderValue > 1200
    ORDER BY AvgOrderValue DESC
""")
print('Products with avg order value above 1200:')
result

Products with avg order value above 1200:


,Product,OrderCount,AvgOrderValue,AvgQuantity,AvgUnitPrice


### 6.5 Months with Revenue Exceeding 50000

In [27]:
# Months where total revenue exceeds 50000
result = run_query("""
    SELECT SUBSTR(Date, 1, 7) AS YearMonth,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue
    FROM orders
    GROUP BY YearMonth
    HAVING TotalRevenue > 50000
    ORDER BY TotalRevenue DESC
""")
print(f'Months with revenue above 50000: {len(result)}')
result

Months with revenue above 50000: 6


,YearMonth,OrderCount,TotalRevenue
0,2024-06,53,68068.54
1,2023-05,49,63836.84
2,2023-01,47,56685.75
3,2023-08,51,54352.14
4,2025-06,49,53047.40
5,2023-10,47,52607.85


## 7. Percentage Contribution Analysis

Calculating the percentage contribution of each category to the overall total is a valuable business metric. This helps identify which segments drive the most revenue.

### 7.1 Revenue Percentage Contribution by Product

In [28]:
# Percentage of total revenue contributed by each product
result = run_query("""
    SELECT Product,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS RevenuePercentage
    FROM orders
    GROUP BY Product
    ORDER BY RevenuePercentage DESC
""")
result

,Product,OrderCount,TotalRevenue,RevenuePercentage
0,Printer,181,195612.61,15.47
1,Chair,178,195620.11,15.47
2,Laptop,173,192126.56,15.19
3,Tablet,179,186568.95,14.75
4,Monitor,163,175651.41,13.89
5,Desk,170,167459.93,13.24
6,Phone,156,151722.39,12.00


### 7.2 Order Percentage Contribution by Order Status

In [29]:
# Percentage of orders in each status
result = run_query("""
    SELECT OrderStatus,
           COUNT(*) AS OrderCount,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS OrderPercentage,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS RevenuePercentage
    FROM orders
    GROUP BY OrderStatus
    ORDER BY OrderCount DESC
""")
result

,OrderStatus,OrderCount,OrderPercentage,TotalRevenue,RevenuePercentage
0,Cancelled,250,20.83,276396.21,21.85
1,Returned,247,20.58,243277.70,19.24
2,Pending,237,19.75,256328.15,20.27
3,Shipped,235,19.58,246159.58,19.46
4,Delivered,231,19.25,242600.32,19.18


### 7.3 Revenue Percentage Contribution by Payment Method

In [30]:
# Revenue share by payment method
result = run_query("""
    SELECT PaymentMethod,
           COUNT(*) AS OrderCount,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS OrderPercentage,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS RevenuePercentage
    FROM orders
    GROUP BY PaymentMethod
    ORDER BY RevenuePercentage DESC
""")
result

,PaymentMethod,OrderCount,OrderPercentage,TotalRevenue,RevenuePercentage
0,Credit Card,234,19.50,263847.63,20.86
1,Online,258,21.50,262442.94,20.75
2,Cash,246,20.50,259786.29,20.54
3,Gift Card,230,19.17,246323.92,19.48
4,Debit Card,232,19.33,232361.18,18.37


### 7.4 Revenue Percentage by Referral Source

In [31]:
# Revenue share by referral source
result = run_query("""
    SELECT ReferralSource,
           COUNT(*) AS OrderCount,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS OrderPercentage,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS RevenuePercentage
    FROM orders
    GROUP BY ReferralSource
    ORDER BY RevenuePercentage DESC
""")
result

,ReferralSource,OrderCount,OrderPercentage,TotalRevenue,RevenuePercentage
0,Instagram,259,21.58,275285.45,21.77
1,Email,250,20.83,261808.55,20.70
2,Google,241,20.08,250441.48,19.80
3,Facebook,228,19.00,250410.90,19.80
4,Referral,222,18.50,226815.58,17.93


### 7.5 Revenue Percentage by Coupon Code

In [32]:
# Revenue share by coupon code
result = run_query("""
    SELECT CouponCode,
           COUNT(*) AS OrderCount,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS OrderPercentage,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(SUM(TotalPrice) * 100.0 / (SELECT SUM(TotalPrice) FROM orders), 2) AS RevenuePercentage
    FROM orders
    GROUP BY CouponCode
    ORDER BY RevenuePercentage DESC
""")
result

,CouponCode,OrderCount,OrderPercentage,TotalRevenue,RevenuePercentage
0,FREESHIP,313,26.08,335036.99,26.49
1,NoCoupon,309,25.75,322401.41,25.49
2,SAVE10,286,23.83,304840.02,24.10
3,WINTER15,292,24.33,302483.54,23.92


## 8. Advanced Queries

These queries combine multiple SQL concepts to answer more complex business questions.

### 8.1 Product Performance by Order Status

In [33]:
# Cross analysis of product and order status
result = run_query("""
    SELECT Product,
           OrderStatus,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue
    FROM orders
    GROUP BY Product, OrderStatus
    ORDER BY Product, OrderCount DESC
""")
result

,Product,OrderStatus,OrderCount,TotalRevenue
0,Chair,Cancelled,45,48660.98
1,Chair,Pending,41,48504.70
2,Chair,Delivered,33,31465.83
3,Chair,Shipped,31,40350.44
4,Chair,Returned,28,26638.16
5,Desk,Pending,38,41390.08
6,Desk,Cancelled,35,39587.69
7,Desk,Shipped,33,33424.89
8,Desk,Returned,32,28831.49
9,Desk,Delivered,32,24225.78


### 8.2 Average Unit Price by Product and Payment Method

In [34]:
# Average unit price for each product across payment methods
result = run_query("""
    SELECT Product,
           PaymentMethod,
           COUNT(*) AS OrderCount,
           ROUND(AVG(UnitPrice), 2) AS AvgUnitPrice,
           ROUND(AVG(TotalPrice), 2) AS AvgTotalPrice
    FROM orders
    GROUP BY Product, PaymentMethod
    HAVING OrderCount >= 10
    ORDER BY Product, AvgTotalPrice DESC
""")
result

,Product,PaymentMethod,OrderCount,AvgUnitPrice,AvgTotalPrice
0,Chair,Credit Card,30,348.70,1180.17
1,Chair,Online,47,342.78,1156.21
2,Chair,Gift Card,36,375.88,1057.34
3,Chair,Cash,34,368.65,1055.96
4,Chair,Debit Card,31,344.20,1029.23
5,Desk,Online,27,340.33,1093.56
6,Desk,Credit Card,37,330.41,1012.81
7,Desk,Gift Card,39,363.46,1012.78
8,Desk,Cash,33,285.36,958.94
9,Desk,Debit Card,34,324.33,862.25


### 8.3 Cancellation Rate by Product

In [35]:
# Cancellation rate for each product
result = run_query("""
    SELECT Product,
           COUNT(*) AS TotalOrders,
           SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) AS CancelledOrders,
           ROUND(SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS CancellationRate,
           SUM(CASE WHEN OrderStatus = 'Returned' THEN 1 ELSE 0 END) AS ReturnedOrders,
           ROUND(SUM(CASE WHEN OrderStatus = 'Returned' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ReturnRate
    FROM orders
    GROUP BY Product
    ORDER BY CancellationRate DESC
""")
result

,Product,TotalOrders,CancelledOrders,CancellationRate,ReturnedOrders,ReturnRate
0,Chair,178,45,25.28,28,15.73
1,Monitor,163,35,21.47,36,22.09
2,Desk,170,35,20.59,32,18.82
3,Laptop,173,35,20.23,39,22.54
4,Phone,156,31,19.87,31,19.87
5,Printer,181,35,19.34,38,20.99
6,Tablet,179,34,18.99,43,24.02


### 8.4 Revenue from Successfully Delivered Orders Only

In [36]:
# Revenue from delivered orders only by product
result = run_query("""
    SELECT Product,
           COUNT(*) AS DeliveredCount,
           ROUND(SUM(TotalPrice), 2) AS DeliveredRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgDeliveredValue,
           ROUND(SUM(TotalPrice) * 100.0 / (
               SELECT SUM(TotalPrice) FROM orders WHERE OrderStatus = 'Delivered'
           ), 2) AS RevenueShareAmongDelivered
    FROM orders
    WHERE OrderStatus = 'Delivered'
    GROUP BY Product
    ORDER BY DeliveredRevenue DESC
""")
result

,Product,DeliveredCount,DeliveredRevenue,AvgDeliveredValue,RevenueShareAmongDelivered
0,Laptop,40,40714.43,1017.86,16.78
1,Phone,38,40345.41,1061.72,16.63
2,Printer,29,38054.73,1312.23,15.69
3,Monitor,31,35999.62,1161.28,14.84
4,Tablet,28,31794.52,1135.52,13.11
5,Chair,33,31465.83,953.51,12.97
6,Desk,32,24225.78,757.06,9.99


### 8.5 Top 10 Highest Spending Customers

In [37]:
# Top 10 customers by total spending
result = run_query("""
    SELECT CustomerID,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalSpent,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           MAX(TotalPrice) AS LargestOrder
    FROM orders
    GROUP BY CustomerID
    ORDER BY TotalSpent DESC
    LIMIT 10
""")
result

,CustomerID,OrderCount,TotalSpent,AvgOrderValue,LargestOrder
0,C38840,2,5723.23,2861.61,3390.95
1,C57276,1,3456.40,3456.40,3456.40
2,C67260,1,3390.80,3390.80,3390.80
3,C13877,1,3384.90,3384.90,3384.90
4,C18404,1,3370.20,3370.20,3370.20
5,C16775,1,3353.75,3353.75,3353.75
6,C65986,1,3352.40,3352.40,3352.40
7,C47778,1,3334.00,3334.00,3334.00
8,C59183,1,3322.55,3322.55,3322.55
9,C25276,1,3313.90,3313.90,3313.90


### 8.6 Quantity Distribution Analysis

In [38]:
# Order distribution by quantity
result = run_query("""
    SELECT Quantity,
           COUNT(*) AS OrderCount,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS Percentage,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue
    FROM orders
    GROUP BY Quantity
    ORDER BY Quantity
""")
result

,Quantity,OrderCount,Percentage,TotalRevenue,AvgOrderValue
0,1,255,21.25,89728.29,351.88
1,2,240,20.00,165729.68,690.54
2,3,237,19.75,261276.18,1102.43
3,4,251,20.92,368090.96,1466.50
4,5,217,18.08,379936.85,1750.86


### 8.7 Items in Cart vs Revenue Analysis

In [39]:
# How does the number of items in cart relate to order value
result = run_query("""
    SELECT ItemsInCart,
           COUNT(*) AS OrderCount,
           ROUND(AVG(TotalPrice), 2) AS AvgOrderValue,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(AVG(Quantity), 2) AS AvgQuantity
    FROM orders
    GROUP BY ItemsInCart
    ORDER BY ItemsInCart
""")
result

,ItemsInCart,OrderCount,AvgOrderValue,TotalRevenue,AvgQuantity
0,1,50,368.68,18434.01,1.00
1,2,78,551.36,43006.24,1.53
2,3,123,722.42,88858.03,2.07
3,4,163,831.14,135475.08,2.47
4,5,191,1055.59,201617.19,2.78
5,6,190,1119.57,212718.75,3.01
6,7,151,1157.72,174815.16,3.42
7,8,129,1488.23,191982.26,3.93
8,9,79,1489.46,117667.19,4.47
9,10,46,1743.22,80188.05,5.00


### 8.8 Coupon Effectiveness: Delivered vs Cancelled

In [40]:
# Success rate by coupon code (Delivered vs Cancelled)
result = run_query("""
    SELECT CouponCode,
           COUNT(*) AS TotalOrders,
           SUM(CASE WHEN OrderStatus = 'Delivered' THEN 1 ELSE 0 END) AS DeliveredOrders,
           SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) AS CancelledOrders,
           ROUND(SUM(CASE WHEN OrderStatus = 'Delivered' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS DeliveryRate,
           ROUND(SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS CancellationRate
    FROM orders
    GROUP BY CouponCode
    ORDER BY DeliveryRate DESC
""")
result

,CouponCode,TotalOrders,DeliveredOrders,CancelledOrders,DeliveryRate,CancellationRate
0,SAVE10,286,63,58,22.03,20.28
1,FREESHIP,313,61,67,19.49,21.41
2,WINTER15,292,55,67,18.84,22.95
3,NoCoupon,309,52,58,16.83,18.77


### 8.9 Yearly Revenue Contribution by Product

In [41]:
# Yearly revenue breakdown by product
result = run_query("""
    SELECT SUBSTR(Date, 1, 4) AS Year,
           Product,
           COUNT(*) AS OrderCount,
           ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
           ROUND(SUM(TotalPrice) * 100.0 / (
               SELECT SUM(TotalPrice) FROM orders o2
               WHERE SUBSTR(o2.Date, 1, 4) = SUBSTR(orders.Date, 1, 4)
           ), 2) AS YearlyRevenueShare
    FROM orders
    GROUP BY Year, Product
    ORDER BY Year, TotalRevenue DESC
""")
result

,Year,Product,OrderCount,TotalRevenue,YearlyRevenueShare
0,2023,Printer,82,95513.73,17.28
1,2023,Chair,80,87585.58,15.85
2,2023,Tablet,77,86678.92,15.68
3,2023,Laptop,74,83231.66,15.06
4,2023,Monitor,71,75876.42,13.73
5,2023,Phone,60,62766.40,11.36
6,2023,Desk,66,60990.53,11.04
7,2024,Chair,70,84785.75,17.66
8,2024,Laptop,63,75611.41,15.74
9,2024,Monitor,63,68450.23,14.25


## 9. Summary Statistics

In [42]:
# Overall business summary
result = run_query("""
    SELECT
        COUNT(*) AS TotalOrders,
        COUNT(DISTINCT CustomerID) AS UniqueCustomers,
        COUNT(DISTINCT Product) AS UniqueProducts,
        ROUND(SUM(TotalPrice), 2) AS GrandTotalRevenue,
        ROUND(AVG(TotalPrice), 2) AS OverallAvgOrderValue,
        MIN(TotalPrice) AS SmallestOrder,
        MAX(TotalPrice) AS LargestOrder,
        SUM(Quantity) AS TotalUnitsSold,
        ROUND(AVG(Quantity), 2) AS AvgQuantityPerOrder,
        MIN(Date) AS EarliestOrder,
        MAX(Date) AS LatestOrder
    FROM orders
""")
print('Overall Business Summary:')
result.T

Overall Business Summary:


,0
TotalOrders,1200
UniqueCustomers,1189
UniqueProducts,7
GrandTotalRevenue,1264761.96
OverallAvgOrderValue,1053.97
SmallestOrder,11.39
LargestOrder,3456.4
TotalUnitsSold,3535
AvgQuantityPerOrder,2.95
EarliestOrder,2023-01-01


In [43]:
# Close the database connection
conn.close()
print('Database connection closed. Analysis complete.')

Database connection closed. Analysis complete.


## Key Findings

**Dataset Overview:** The dataset contains 1200 orders spanning from January 2023 to June 2025 across 7 product categories with nearly 1189 unique customers.

**SQL Skills Demonstrated:**
- **SELECT with WHERE:** Filtered orders by price thresholds and product types and order statuses and date ranges and multiple combined conditions
- **ORDER BY:** Sorted results by single and multiple columns in ascending and descending order
- **GROUP BY with Aggregations:** Used COUNT and SUM and AVG and MIN and MAX to summarize data by product and status and payment method and referral source and coupon code and time period
- **HAVING:** Filtered grouped results to identify high revenue products and popular payment methods and repeat customers and high performing months
- **Percentage Contributions:** Calculated revenue and order percentage shares using subqueries for products and statuses and payment methods and referral sources and coupon codes
- **Advanced Queries:** Performed cross tabulations and cancellation rate analysis and conditional aggregation with CASE WHEN and correlated subqueries for yearly product revenue shares